## Scaled Dot-Product Attention

In [2]:
import numpy as np
import math

def softmax(x):
    # x: (30, 8, 200, 200)
    x = x - np.max(x, axis=-1, keepdims=True)
    exp = np.exp(x)
    return exp / np.sum(exp, axis=-1, keepdims=True)


def scaled_dot_product(q, k, v, mask=None):

    # q, k, v:
    # (batch_size, num_heads, seq_len, head_dim)
    # (30, 8, 200, 64)

    d_k = q.shape[-1]

    # k transpose:
    # (30, 8, 64, 200)

    scaled = np.matmul(q, np.transpose(k, (0,1,3,2))) / math.sqrt(d_k)

    # scaled:
    # (30, 8, 200, 200)
    # Each token attends to 200 tokens

    if mask is not None:
        # mask: (200, 200)
        scaled += mask[np.newaxis, np.newaxis, :, :]

    attention = softmax(scaled)

    # attention:
    # (30, 8, 200, 200)

    values = np.matmul(attention, v)

    # values:
    # (30, 8, 200, 64)

    return values

## Masked Multi-Head Self Attention (Decoder First Block)

In [3]:
class MultiHeadSelfAttention:

    def __init__(self, d_model, num_heads):

        self.d_model = d_model              # 512
        self.num_heads = num_heads          # 8
        self.head_dim = d_model // num_heads  # 64

        # (512 → 1536)
        self.W_qkv = np.random.randn(d_model, 3*d_model)

        # Final projection (512 → 512)
        self.W_o = np.random.randn(d_model, d_model)

    def forward(self, x, mask=None):

        # x:
        # (30, 200, 512)

        batch_size, seq_len, _ = x.shape

        qkv = np.matmul(x, self.W_qkv)

        # qkv:
        # (30, 200, 1536)

        qkv = qkv.reshape(batch_size,
                          seq_len,
                          self.num_heads,
                          3*self.head_dim)

        # (30, 200, 8, 192)

        qkv = np.transpose(qkv, (0,2,1,3))

        # (30, 8, 200, 192)

        q, k, v = np.split(qkv, 3, axis=-1)

        # q,k,v:
        # (30, 8, 200, 64)

        values = scaled_dot_product(q, k, v, mask)

        # values:
        # (30, 8, 200, 64)

        values = np.transpose(values, (0,2,1,3))

        # (30, 200, 8, 64)

        values = values.reshape(batch_size, seq_len, self.d_model)

        # (30, 200, 512)

        out = np.matmul(values, self.W_o)

        # Final output:
        # (30, 200, 512)

        return out

## Cross Attention (Encoder–Decoder)

In [4]:
class MultiHeadCrossAttention:

    def __init__(self, d_model, num_heads):

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # Separate projections
        self.W_q = np.random.randn(d_model, d_model)
        self.W_kv = np.random.randn(d_model, 2*d_model)
        self.W_o = np.random.randn(d_model, d_model)

    def forward(self, encoder_output, decoder_input):

        # encoder_output:
        # (30, 200, 512)

        # decoder_input:
        # (30, 200, 512)

        batch_size, seq_len, _ = encoder_output.shape

        q = np.matmul(decoder_input, self.W_q)

        # (30, 200, 512)

        kv = np.matmul(encoder_output, self.W_kv)

        # (30, 200, 1024)

        q = q.reshape(batch_size, seq_len,
                      self.num_heads, self.head_dim)

        kv = kv.reshape(batch_size, seq_len,
                        self.num_heads, 2*self.head_dim)

        # q:  (30, 200, 8, 64)
        # kv: (30, 200, 8, 128)

        q = np.transpose(q, (0,2,1,3))
        kv = np.transpose(kv, (0,2,1,3))

        # q:  (30, 8, 200, 64)
        # kv: (30, 8, 200, 128)

        k, v = np.split(kv, 2, axis=-1)

        # k,v:
        # (30, 8, 200, 64)

        values = scaled_dot_product(q, k, v)

        # (30, 8, 200, 64)

        values = np.transpose(values, (0,2,1,3))

        # (30, 200, 8, 64)

        values = values.reshape(batch_size, seq_len, self.d_model)

        # (30, 200, 512)

        out = np.matmul(values, self.W_o)

        # (30, 200, 512)

        return out

## Layer Normalization

In [5]:
class LayerNormalization:

    def __init__(self, d_model):
        self.gamma = np.ones(d_model)
        self.beta = np.zeros(d_model)

    def forward(self, x):

        # x:
        # (30, 200, 512)

        mean = np.mean(x, axis=-1, keepdims=True)

        # (30, 200, 1)

        var = np.mean((x - mean)**2, axis=-1, keepdims=True)

        # (30, 200, 1)

        std = np.sqrt(var + 1e-5)

        y = (x - mean) / std

        # (30, 200, 512)

        out = self.gamma * y + self.beta

        # (30, 200, 512)

        return out

## FeedForward

In [6]:
class PositionwiseFeedForward:

    def __init__(self, d_model, hidden):

        # 512 → 2048
        self.W1 = np.random.randn(d_model, hidden)

        # 2048 → 512
        self.W2 = np.random.randn(hidden, d_model)

    def forward(self, x):

        # x:
        # (30, 200, 512)

        x = np.matmul(x, self.W1)

        # (30, 200, 2048)

        x = np.maximum(0, x)

        x = np.matmul(x, self.W2)

        # (30, 200, 512)

        return x

## Decoder Layer

In [7]:
class DecoderLayer:

    def __init__(self, d_model, ffn_hidden, num_heads):

        self.self_attention = MultiHeadSelfAttention(d_model, num_heads)
        self.norm1 = LayerNormalization(d_model)

        self.cross_attention = MultiHeadCrossAttention(d_model, num_heads)
        self.norm2 = LayerNormalization(d_model)

        self.ffn = PositionwiseFeedForward(d_model, ffn_hidden)
        self.norm3 = LayerNormalization(d_model)

    def forward(self, encoder_output, decoder_input, mask):

        # decoder_input:
        # (30, 200, 512)

        # 1️⃣ Masked Self Attention
        residual = decoder_input

        y = self.self_attention.forward(decoder_input, mask)

        # (30, 200, 512)

        y = self.norm1.forward(y + residual)

        # (30, 200, 512)

        # 2️⃣ Cross Attention
        residual = y

        y = self.cross_attention.forward(encoder_output, y)

        # (30, 200, 512)

        y = self.norm2.forward(y + residual)

        # (30, 200, 512)

        # 3️⃣ Feed Forward
        residual = y

        y = self.ffn.forward(y)

        # (30, 200, 512)

        y = self.norm3.forward(y + residual)

        # (30, 200, 512)

        return y

## Decoder stack on each other

In [8]:
class Decoder:

    def __init__(self, d_model, ffn_hidden, num_heads, num_layers):

        self.layers = [
            DecoderLayer(d_model, ffn_hidden, num_heads)
            for _ in range(num_layers)
        ]

    def forward(self, encoder_output, decoder_input, mask):

        y = decoder_input

        # (30, 200, 512)

        for layer in self.layers:
            y = layer.forward(encoder_output, y, mask)

        # Final:
        # (30, 200, 512)

        return y